# 02 Prompt, Structured Output, and Function Calling

**Goal for this notebook:** learn how to move from free-form model text to program-friendly structured data and simple tool calls.

This notebook focuses on one workflow:

```text
prompt -> JSON-shaped text -> schema-validated output -> routing decision
```

Optional preview:

```text
routing decision -> tool selection -> local function execution
```

## Suggested Pace

1. **Prompting for shape**: ask the model to return JSON and see why prompt-only control is fragile.
2. **JSON mode**: ask the API to return valid JSON.
3. **JSON schema + Pydantic**: make the output match a schema and validate it in Python.
4. **Mini task router**: classify user requests into structured routes.
5. **Optional preview**: function calling and local tool execution.

## What Is Optional

Section 5 is an optional preview. If time is short, finish the structured output router first and save function calling for a later tools/agent lesson.

## References

- [OpenAI Structured Outputs guide](https://platform.openai.com/docs/guides/structured-outputs)
- [OpenAI Function Calling guide](https://platform.openai.com/docs/guides/function-calling)
- [OpenAI Responses API reference](https://developers.openai.com/api/reference/responses/overview)
- [Pydantic documentation](https://docs.pydantic.dev/)
- [JSON Schema documentation](https://json-schema.org/learn/getting-started-step-by-step)

## Setup Check

Run the next cell first. It loads `.env`, creates an OpenAI client, and selects the model for this notebook.

提示：这个 notebook 是 self-contained 的，不会 import `src/` 里的任何作业代码。

In [ ]:
import json
import os
from typing import Literal

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv()

if os.getenv("OPENAI_BASE_URL") == "":
    os.environ.pop("OPENAI_BASE_URL", None)

MODEL = os.getenv("LLM_MODEL", "gpt-5.4-mini")
client = OpenAI()

print("OPENAI_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL") or "<default OpenAI endpoint>")
print("MODEL:", MODEL)

---

## Section 1 · Prompting for Shape

A normal prompt can ask the model to return JSON. This is useful, but it is still only a natural-language instruction.

The model may return valid JSON, but it might also add extra text, miss fields, or choose unexpected field names.

提示：prompt 可以提高概率，但 prompt 本身不是 schema，也不是 validation。

In [ ]:
prompt_only = client.responses.create(
    model=MODEL,
    input="""
Classify this request and return JSON with keys: task_type, confidence, reason.

Request: Please summarize this article into three bullet points.
""",
    instructions="You are a task router. Return JSON only.",
    temperature=0.2,
    max_output_tokens=300,
)

print(prompt_only.output_text)

In [ ]:
try:
    parsed_prompt_only = json.loads(prompt_only.output_text)
    print(json.dumps(parsed_prompt_only, indent=2, ensure_ascii=False))
except json.JSONDecodeError as exc:
    print("Could not parse model output as JSON:", exc)

### Why Prompt-Only JSON Is Not Enough

Prompt-only JSON is better than free-form text, but your program still has to ask:

- Is it valid JSON?
- Are all required fields present?
- Are field names stable?
- Are values in the allowed set?
- Are types correct?

This is why structured output and validation matter.

---

## Section 2 · JSON Mode

JSON mode asks the API to return syntactically valid JSON.

To use JSON mode with the Responses API, you need two things:

1. The request `input` must mention `json` in some form.
2. The API call must set `text={"format": {"type": "json_object"}}`.

It is useful for simple demos and quick prototypes, but it does **not** guarantee that the output matches your exact schema.

提示：如果你设置了 `json_object`，但 input 里没有出现 `json`，OpenAI API 会直接报错。

In [ ]:
json_mode_response = client.responses.create(
    model=MODEL,
    # Requirement 1: the input must mention "JSON".
    input="Return a JSON object that classifies this request: translate this sentence into Spanish.",
    instructions="Return JSON only with keys: task_type, confidence, reason.",
    # Requirement 2: set text.format.type to json_object.
    text={"format": {"type": "json_object"}},
    temperature=0.2,
    max_output_tokens=300,
)

print(json_mode_response.output_text)

In [ ]:
parsed_json_mode = json.loads(json_mode_response.output_text)
print(json.dumps(parsed_json_mode, indent=2, ensure_ascii=False))

### JSON Mode vs Schema

JSON mode answers one question:

```text
Is the output valid JSON?
```

It does not fully answer:

```text
Does this JSON match the shape my program expects?
```

So JSON mode is best understood as a syntax constraint, not a full data contract.

For a full data contract, use JSON schema and validate the result in Python.

---

## Section 3 · Structured Output with JSON Schema and Pydantic

Now define the shape we actually want.

### What Is Pydantic?

Pydantic is a Python library for defining data models and validating data.

In this notebook, we use Pydantic for two things:

- Define the shape of the data we want from the model
- Validate that the model output really matches that shape

You can think of a Pydantic model as a Python class that also knows how to check input data.

Pydantic gives us a Python model. JSON schema gives the API a machine-readable contract. Together, they help turn model output into data your program can trust.

提示：JSON schema 是给 API 看的结构约束；Pydantic 是 Python 代码里真正验证和使用这个结构的地方。

In [ ]:
class RouteDecision(BaseModel):
    task_type: Literal["chat", "summarize", "translate", "code_help", "tool_call"]
    confidence: float = Field(ge=0, le=1)
    reason: str


schema = RouteDecision.model_json_schema()
schema["additionalProperties"] = False

print(json.dumps(schema, indent=2))

In [ ]:
structured_response = client.responses.create(
    model=MODEL,
    input="Classify this request: Can you help me debug this Python error?",
    instructions="Classify the user request into the RouteDecision schema.",
    text={
        "format": {
            "type": "json_schema",
            "name": "route_decision",
            "schema": schema,
            "strict": True,
        }
    },
    temperature=0.1,
    max_output_tokens=300,
)

print(structured_response.output_text)

In [ ]:
decision = RouteDecision.model_validate_json(structured_response.output_text)
print(decision)
print("task_type:", decision.task_type)
print("confidence:", decision.confidence)
print("reason:", decision.reason)

In [ ]:
bad_payload = {
    "task_type": "debug",  # not in the allowed Literal values
    "confidence": 1.2,     # outside the 0-1 range
    "reason": "This intentionally violates the schema.",
}

try:
    RouteDecision.model_validate(bad_payload)
except ValidationError as exc:
    print(exc)

---

## Section 4 · Mini Task Router

A task router classifies a user request before deciding what your application should do next.

This is a common agent pattern:

```text
user request -> structured classification -> application branch
```

The important design idea is that the model is not solving the whole task yet. It is making a small structured decision that your program can use.

In this section, `route_request(...)` will:

1. Send the user request to the model.
2. Ask for output matching the `RouteDecision` schema.
3. Validate the response with Pydantic.
4. Return a typed `RouteDecision` object.

In a real system, each route could trigger a different workflow.

In [ ]:
def route_request(user_request: str) -> RouteDecision:
    """Classify a user request into a structured route decision.

    The model returns JSON that should match the RouteDecision schema.
    Pydantic validates the JSON and converts it into a typed Python object.
    """

    response = client.responses.create(
        model=MODEL,
        input=f"Classify this request: {user_request}",
        instructions="Classify the user request into the RouteDecision schema.",
        text={
            "format": {
                "type": "json_schema",
                "name": "route_decision",
                "schema": schema,
                "strict": True,
            }
        },
        temperature=0.1,
        max_output_tokens=300,
    )
    return RouteDecision.model_validate_json(response.output_text)


examples = [
    "Please summarize this meeting transcript.",
    "Translate this sentence into Chinese: good morning.",
    "Can you explain what a Python decorator is?",
    "Help me fix this stack trace.",
    "Count how many words are in this sentence.",
]

for request in examples:
    decision = route_request(request)
    print(f"Request: {request}")
    print(f"Route  : {decision.task_type} ({decision.confidence:.2f})")
    print(f"Reason : {decision.reason}\n")

### Router Takeaway

A router does not solve the user's task by itself. It decides which path the application should take next.

Examples:

- `summarize` -> call a summarization workflow
- `translate` -> call a translation workflow
- `code_help` -> use a coding assistant prompt
- `tool_call` -> ask the model to select a function/tool

The docstring on `route_request(...)` is for developers. It explains what the function does, what it returns, and why validation matters. Later, tool descriptions play a similar role for the model.

提示：agent 系统里很多“智能”其实来自 routing + structured decisions，而不是一次性让模型自由发挥。

---

## Section 5 · Optional Preview: Function Calling

This section is a preview for Class 4. In Class 4, we will focus on function calling, tool execution loops, and tool-using agents in detail.

Function calling lets the model choose a tool and provide structured arguments.

This section is optional for Lesson 2.

The model does **not** execute Python code by itself. Your program still owns execution:

```text
model chooses tool + arguments -> Python executes function -> result can be shown or sent back
```

For this preview, use two small local tools defined inside the notebook.

In [ ]:
def get_word_count(text: str) -> int:
    return len(text.split())


def uppercase(text: str) -> str:
    return text.upper()


LOCAL_TOOLS = {
    "get_word_count": get_word_count,
    "uppercase": uppercase,
}

In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_word_count",
        "description": "Count the number of whitespace-delimited words in a text string.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to count words in."}
            },
            "required": ["text"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "uppercase",
        "description": "Convert text to uppercase.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to convert."}
            },
            "required": ["text"],
            "additionalProperties": False,
        },
    },
]

print(json.dumps(tools, indent=2))

In [ ]:
tool_response = client.responses.create(
    model=MODEL,
    input="How many words are in this text: structured output makes agents easier to test",
    instructions="Use a tool when it is helpful. Do not answer directly if a tool can solve the task.",
    tools=tools,
    temperature=0.1,
    max_output_tokens=300,
)

print(json.dumps(tool_response.model_dump(), indent=2, ensure_ascii=False, default=str))

In [ ]:
function_calls = [item for item in tool_response.output if item.type == "function_call"]

for call in function_calls:
    print("Tool name:", call.name)
    print("Arguments:", call.arguments)

In [ ]:
for call in function_calls:
    arguments = json.loads(call.arguments)
    result = LOCAL_TOOLS[call.name](**arguments)
    print(f"{call.name}({arguments}) -> {result}")

### Function Calling Takeaway

Function calling separates decision-making from execution:

- The model decides which function to call.
- The model provides structured arguments.
- Your Python code validates and executes the function.
- Your application decides what to do with the result.

提示：function calling 不是“模型真的会执行函数”，而是模型输出一个结构化的工具调用请求。

---

## Checkpoint · Core Path Complete

At this point, you have completed the main notebook path:

- You saw why prompt-only JSON is fragile.
- You used JSON mode to request valid JSON.
- You defined a Pydantic model and JSON schema.
- You validated model output with Pydantic.
- You built a mini task router with structured output.

## Light Assignment

Run the core path from top to bottom and make sure it works:

1. The setup check can read your OpenAI API key.
2. JSON mode returns parseable JSON.
3. The structured router returns a valid `RouteDecision`.
4. Optional stretch: run Section 5 and observe how function calling selects a local tool.

## Congratulations · Second Class Complete

You have finished the second notebook core path.

In this lesson, you learned how to:

- Use prompts to ask for a specific output shape
- Use JSON mode for valid JSON output
- Use JSON schema for stricter structured output
- Validate model output with Pydantic
- Build a simple task router

If you also finished the optional preview, you saw how function calling lets the model choose a local tool and provide arguments.

提示：agent 系统的核心不是让模型自由发挥，而是让模型输出可验证的结构化决策，然后由程序安全地执行下一步。